In [3]:
# Cell 2: Imports and utility functions
import torch
import torch.profiler
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
import lpips
import kornia
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

to_tensor = T.Compose([
    T.ToTensor(),  # Converts to [0,1]
    T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # Normalize to [-1,1] for LPIPS
])

to_pil = T.Compose([
    T.Normalize(mean=[-1, -1, -1], std=[2, 2, 2]),
    T.ToPILImage()
])

In [4]:
import torch
import torch.profiler
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
import lpips
import kornia
import matplotlib.pyplot as plt

target_path = 'test-images/mac.png'     
can_path = 'Logos/chiquita-logo.png'  

# === Configuration: Set your real-world sizes ===
canvas_width_in = 8.5     # canvas width in inches
canvas_height_in = 10.0    # canvas height in inches
can_width_in = 1.0        # can width in inches
can_height_in = 1.0       # can height in inches
dpi = 45                   # dots per inch (resolution)
downscale_factor = 1.0    # apply 25% scaling for optimization

# === Convert inches to pixels ===
canvas_width_px = int(canvas_width_in * dpi)
canvas_height_px = int(canvas_height_in * dpi)
can_width_px = int(can_width_in * dpi)
can_height_px = int(can_height_in * dpi)

print(f"Canvas: {canvas_width_px} x {canvas_height_px} px")
print(f"Each can: {can_width_px} x {can_height_px} px")

# === Load and resize target/can image ===
target_pil = Image.open(target_path).convert('RGB').resize((canvas_width_px, canvas_height_px))
can_pil = Image.open(can_path).convert('RGBA').resize((can_width_px, can_height_px))

# === Convert to full-resolution tensors ===
target_tensor_full = to_tensor(target_pil).unsqueeze(0).to(device)
can_rgb_full = to_tensor(can_pil.convert('RGB')).unsqueeze(0).to(device)
can_alpha_full = T.ToTensor()(can_pil.split()[-1]).unsqueeze(0).to(device)

# === Downscale all tensors for optimization ===
def downscale(tensor, factor):
    return F.interpolate(tensor, scale_factor=factor, mode='bilinear', align_corners=False)

target_tensor = downscale(target_tensor_full, downscale_factor)
can_rgb = downscale(can_rgb_full, downscale_factor)
can_alpha = F.interpolate(can_alpha_full, scale_factor=downscale_factor, mode='bilinear', align_corners=False)

# === Create blank canvas at downscaled resolution ===
canvas = torch.zeros_like(target_tensor).detach()


Canvas: 382 x 450 px
Each can: 45 x 45 px


In [13]:
import lpips
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
lpips_model = lpips.LPIPS(net='alex').to(device)


Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


C:\Users\Eoin\AppData\Roaming\Python\Python312\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\Eoin\AppData\Roaming\Python\Python312\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: C:\Users\Eoin\AppData\Roaming\Python\Python312\site-packages\lpips\weights\v0.1\alex.pth


C:\Users\Eoin\AppData\Roaming\Python\Python312\site-packages\lpips\lpips.py:107: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.load_state_dict(torch.load(model_path, ma

In [25]:
import json
import pandas as pd

def export_placements(individual, output_path="can_placements"):
    array = individual.detach().cpu().numpy()
    data = [
        {
            "x": float(row[0]),
            "y": float(row[1]),
            "rotation": float(row[2]),
            "layer": float(row[3]),
            "can_index": int(round(row[4]))
        }
        for row in array
    ]

    # Save to CSV
    csv_path = f"{output_path}.csv"
    pd.DataFrame(data).to_csv(csv_path, index=False)

    # Save to JSON
    json_path = f"{output_path}.json"
    with open(json_path, "w") as f:
        json.dump(data, f, indent=2)

    print(f"Exported placements to {csv_path} and {json_path}")

In [49]:
import os
from PIL import Image
import torchvision.transforms as T
import torchvision.transforms.functional as TF
import torch.nn.functional as F
import torch
import random
import numpy as np

# === CONFIGURABLE PARAMETERS ===
POP_SIZE = 200
GENERATIONS = 1000
NUM_CANS = 126
MUTATION_RATE = 0.6

canvas_width_in = 4.25
canvas_height_in = 5.0
can_width_in = 1.0
can_height_in = 1.0
dpi = 45
downscale_factor = 1.0  # for faster optimization
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# === Utility: Convert PIL to torch tensor ===
def load_and_prepare_image(path, size, mode='RGB'):
    pil = Image.open(path).convert(mode).resize(size)
    return T.ToTensor()(pil).unsqueeze(0).to(device)

# === Load all can images from a folder ===
def load_can_images(folder_path, can_size_px):
    can_images = []
    for fname in os.listdir(folder_path):
        if fname.lower().endswith(('.png', '.jpg', '.jpeg')):
            full_path = os.path.join(folder_path, fname)
            pil = Image.open(full_path).convert("RGBA").resize(can_size_px)
            rgb = T.ToTensor()(pil.convert("RGB")).unsqueeze(0).to(device)
            alpha = T.ToTensor()(pil.split()[-1]).unsqueeze(0).to(device)
            can_images.append((rgb, alpha))
    if not can_images:
        raise ValueError("No valid images found in the provided can folder.")
    return can_images

# === Downscale helper ===
def downscale(tensor, factor):
    return F.interpolate(tensor, scale_factor=factor, mode='bilinear', align_corners=False)

# === Wrapper: optimize cans given folder + image path ===
def run_ga_from_user_input(cans_folder_path, reference_image_path, background_color=(0, 0, 0)):
    canvas_width_px = int(canvas_width_in * dpi)
    canvas_height_px = int(canvas_height_in * dpi)
    can_width_px = int(can_width_in * dpi)
    can_height_px = int(can_height_in * dpi)
    can_size_px = (can_width_px, can_height_px)

    print(f"\nCanvas: {canvas_width_px} x {canvas_height_px} px")
    print(f"Each can: {can_width_px} x {can_height_px} px")

    # Load target image and scale
    target_pil = Image.open(reference_image_path).convert("RGB").resize((canvas_width_px, canvas_height_px))
    target_tensor_full = T.ToTensor()(target_pil).unsqueeze(0).to(device)

    # Load all can images
    can_images_full = load_can_images(cans_folder_path, can_size_px)

    # Downscale for optimization
    target_tensor = downscale(target_tensor_full, downscale_factor)
    can_images = [(downscale(rgb, downscale_factor), F.interpolate(alpha, scale_factor=downscale_factor, mode='bilinear', align_corners=False))
                  for rgb, alpha in can_images_full]

    # Run optimization
    best_individual = optimize_with_ga(target_tensor, can_images, background_color=background_color, save_progress=True, output_dir="progress_images")



    # Render final canvas
    final_canvas = render_canvas(best_individual, can_images, target_tensor.shape)
    result_image = TF.to_pil_image(final_canvas.squeeze(0).clamp(0, 1).cpu())
    result_image.save("optimized_result.png")
    print("Saved final result as optimized_result.png")

    # Final render
    final_canvas = render_canvas(best_individual, can_images, target_tensor.shape)
    result_image = TF.to_pil_image(final_canvas.squeeze(0).clamp(0, 1).cpu())
    result_image.save("optimized_result.png")

    # Save placement data
    export_placements(best_individual)



# === Render a full canvas from an individual ===
def render_canvas(individual, can_images, canvas_shape, background_color=(0, 0, 0)):
    # Initialize canvas with background color
    r, g, b = [c / 255.0 for c in background_color]
    bg_tensor = torch.tensor([r, g, b], device=device).view(1, 3, 1, 1)
    canvas = bg_tensor.expand(1, 3, canvas_shape[2], canvas_shape[3]).clone()

    sorted_idx = torch.argsort(individual[:, 3])  # sort by layer (4th column)

    for i in sorted_idx:
        x, y, angle, _, can_idx = individual[i]
        x, y = int(x.item()), int(y.item())
        can_idx = int(torch.clamp(can_idx, 0, len(can_images)-1).item())

        can_rgb, can_alpha = can_images[can_idx]

        # Rotate can
        rotated_rgb = TF.rotate(can_rgb[0], angle.item(), fill=(0, 0, 0))
        rotated_alpha = TF.rotate(can_alpha[0], angle.item(), fill=0)

        h, w = rotated_rgb.shape[1:]
        x_end = min(x + w, canvas.shape[3])
        y_end = min(y + h, canvas.shape[2])
        x_start = max(x, 0)
        y_start = max(y, 0)

        # 🛡️ Skip if completely out of bounds
        if x_start >= x_end or y_start >= y_end:
            continue

        # Crop image and alpha to fit canvas
        rgb_crop = rotated_rgb[:, y_start - y:y_end - y, x_start - x:x_end - x]
        alpha_crop = rotated_alpha[:, y_start - y:y_end - y, x_start - x:x_end - x]

        region = canvas[0, :, y_start:y_end, x_start:x_end]
        canvas[0, :, y_start:y_end, x_start:x_end] = alpha_crop * rgb_crop + (1 - alpha_crop) * region

    return canvas


# === Fitness: negative LPIPS score ===
def fitness(individual, target_tensor, can_images):
    rendered = render_canvas(individual, can_images, target_tensor.shape)
    with torch.no_grad():
        return -lpips_model(rendered, target_tensor).item()

# === Crossover two parents ===
def crossover(p1, p2):
    mask = torch.rand_like(p1) > 0.5
    return torch.where(mask, p1, p2)

# === Mutate an individual ===
def mutate(individual, num_can_types):
    mutation_strength = torch.tensor([5.0, 5.0, 15.0, 0.2, 2.0], device=device)
    mutation = (torch.rand_like(individual) - 0.5) * 2 * mutation_strength
    individual += mutation
    individual[:, 0] = individual[:, 0].clamp(0, canvas_width_px - can_width_px)
    individual[:, 1] = individual[:, 1].clamp(0, canvas_height_px - can_height_px)
    individual[:, 2] = individual[:, 2].clamp(-180, 180)
    individual[:, 3] = individual[:, 3].clamp(0, 1)
    individual[:, 4] = individual[:, 4].clamp(0, num_can_types - 1)  # use passed-in value
    return individual

# === Main GA loop ===
def optimize_with_ga(target_tensor, can_images, background_color=(0, 0, 0), save_progress=False, output_dir="progress_images"):

    import os
    if save_progress:
        os.makedirs(output_dir, exist_ok=True)

    num_can_types = len(can_images)

    population = [generate_individual(num_can_types) for _ in range(POP_SIZE)]

    for gen in range(GENERATIONS):
        scores = [fitness(ind, target_tensor, can_images) for ind in population]
        best_score = max(scores)
        best_ind = population[np.argmax(scores)]
        print(f"Generation {gen}: LPIPS = {-best_score:.4f}")

        if save_progress and gen % 10 == 0:
            canvas_snapshot = render_canvas(best_ind, can_images, target_tensor.shape, background_color)
            img = TF.to_pil_image(canvas_snapshot.squeeze(0).clamp(0, 1).cpu())
            img.save(os.path.join(output_dir, f"gen_{gen:03d}.png"))

        # === Elitism + Diversity ===
        elite_size = max(1, POP_SIZE // 10)
        num_random = elite_size  # inject diversity
        num_offspring = POP_SIZE - elite_size - num_random

        # Sort population by fitness
        sorted_indices = np.argsort(scores)
        elites = [population[i] for i in sorted_indices[-elite_size:]]
        top_half = [population[i] for i in sorted_indices[-POP_SIZE // 2:]]

        # Generate offspring
        new_offspring = []
        while len(new_offspring) < num_offspring:
            p1, p2 = random.sample(top_half, 2)
            child = crossover(p1, p2)
            if random.random() < MUTATION_RATE:
                child = mutate(child, num_can_types)
            new_offspring.append(child)

        # Inject new random individuals for diversity
        new_random = [generate_individual(num_can_types) for _ in range(num_random)]

        # Form next generation
        population = elites + new_offspring + new_random

    final_scores = [fitness(ind, target_tensor, can_images) for ind in population]
    return population[np.argmax(final_scores)]

# === To visualize result later ===
def tensor_to_pil(t):
    return TF.to_pil_image(t.squeeze(0).cpu().clamp(0, 1))

def generate_individual(num_can_types):
    x = torch.rand(NUM_CANS) * (canvas_width_px - can_width_px)
    y = torch.rand(NUM_CANS) * (canvas_height_px - can_height_px)
    r = torch.rand(NUM_CANS) * 360 - 180
    l = torch.rand(NUM_CANS)
    i = torch.randint(0, num_can_types, (NUM_CANS,), dtype=torch.float32)
    return torch.stack([x, y, r, l, i], dim=1).to(device)





In [50]:

run_ga_from_user_input("logos_picked", "test-images/mac.png", background_color=(201, 207, 207))




Canvas: 191 x 225 px
Each can: 45 x 45 px
Generation 0: LPIPS = 0.5458
Generation 1: LPIPS = 0.5454
Generation 2: LPIPS = 0.5433
Generation 3: LPIPS = 0.5416
Generation 4: LPIPS = 0.5416
Generation 5: LPIPS = 0.5414
Generation 6: LPIPS = 0.5348
Generation 7: LPIPS = 0.5348
Generation 8: LPIPS = 0.5348
Generation 9: LPIPS = 0.5348
Generation 10: LPIPS = 0.5348
Generation 11: LPIPS = 0.5321
Generation 12: LPIPS = 0.5243
Generation 13: LPIPS = 0.5243
Generation 14: LPIPS = 0.5193
Generation 15: LPIPS = 0.5193
Generation 16: LPIPS = 0.5094
Generation 17: LPIPS = 0.5094
Generation 18: LPIPS = 0.5094
Generation 19: LPIPS = 0.5094
Generation 20: LPIPS = 0.5094
Generation 21: LPIPS = 0.5094
Generation 22: LPIPS = 0.5094
Generation 23: LPIPS = 0.5094
Generation 24: LPIPS = 0.5094
Generation 25: LPIPS = 0.5094
Generation 26: LPIPS = 0.5094
Generation 27: LPIPS = 0.5094
Generation 28: LPIPS = 0.5094
Generation 29: LPIPS = 0.5094
Generation 30: LPIPS = 0.5094
Generation 31: LPIPS = 0.5094
Generat